Module 6: Image explanation (vision model)

ollama pull qwen2.5vl:3b
python -m pip install ollama

In [5]:
%pip install ollama pytesseract opencv-python numpy pillow

Note: you may need to restart the kernel to use updated packages.


In [9]:
!ollama list


NAME               ID              SIZE      MODIFIED     
gemma4:latest      dc35e8d9c606    6.6 GB    24 hours ago    
llama3.2:latest    a80c4f17acd5    2.0 GB    7 days ago      


In [10]:
!ollama pull qwen2.5vl:3b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling e9758e589d44:   0% ▕                  ▏ 1.8 MB/3.2 GB                  pulling manifest 
pulling e9758e589d44:   0% ▕                  ▏ 3.2 MB/3.2 GB                  pulling manifest 
pulling e9758e589d44:   0% ▕                  ▏ 3.6 MB/3.2 GB     

In [11]:
!ollama list

NAME               ID              SIZE      MODIFIED           
qwen2.5vl:3b       fb90415cde1e    3.2 GB    About a minute ago    
gemma4:latest      dc35e8d9c606    6.6 GB    25 hours ago          
llama3.2:latest    a80c4f17acd5    2.0 GB    7 days ago            


In [12]:
MODEL = "qwen2.5vl:3b"

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain this image.",
            "images": [str(image_path)]
        }
    ]
)

print(response["message"]["content"])

The image shows a person standing outdoors, leaning against a low wall or barrier. The background features a large, open area that appears to be a flooded or waterlogged field, with a mountainous landscape in the distance. The person is wearing a blue and red checkered shirt. The overall scene suggests a natural, possibly rural or semi-rural setting.


In [13]:

from pathlib import Path
import cv2
import numpy as np
import pytesseract
import ollama

MODEL = "qwen2.5vl:3b"

# Set this path if Tesseract is installed here.
tesseract_exe = Path(
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)
if tesseract_exe.is_file():
    pytesseract.pytesseract.tesseract_cmd = str(tesseract_exe)


def extract_text(path):
    """Load an image, preprocess it, and extract text."""
    path = Path(path).expanduser()

    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")

    image = cv2.imread(str(path))
    if image is None:
        raise ValueError(f"Cannot read image: {path}")

    h, w = image.shape[:2]
    if w < 1400:
        scale = 1400 / w
        image = cv2.resize(
            image, None, fx=scale, fy=scale,
            interpolation=cv2.INTER_CUBIC
        )

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    text = pytesseract.image_to_string(
        gray, lang="eng", config="--oem 3 --psm 6"
    ).strip()

    data = pytesseract.image_to_data(
        gray,
        lang="eng",
        config="--oem 3 --psm 6",
        output_type=pytesseract.Output.DICT
    )

    scores = []
    for word, conf in zip(data["text"], data["conf"]):
        try:
            score = float(conf)
            if word.strip() and score >= 0:
                scores.append(score)
        except (ValueError, TypeError):
            pass

    confidence = sum(scores) / len(scores) if scores else 0.0
    return text, confidence, "tesseract"


def explain_image(path, question="Explain this image."):
    path = Path(path).expanduser()

    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")

    ocr_text, confidence, engine = extract_text(path)

    prompt = f"""
You are an accurate image understanding assistant.

User question: {question}

OCR engine: {engine}
OCR confidence: {confidence:.1f}%

OCR text:
{ocr_text or "(No text detected)"}

Analyze the image itself as well. Explain visible objects, text,
tables, charts, and diagrams. Do not invent unreadable details.
"""

    response = ollama.chat(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": prompt,
            "images": [str(path)]
        }]
    )

    return response["message"]["content"], ocr_text, confidence, engine


# Run in Jupyter Notebook
image_path = input("Enter your image path: ").strip().strip('"')
question = input("Ask a question about the image: ").strip()

try:
    answer, text, confidence, engine = explain_image(
        image_path,
        question or "Explain this image."
    )

    print(f"\nOCR engine: {engine}")
    print(f"Confidence: {confidence:.1f}%")
    print("\nEXTRACTED TEXT:\n", text)
    print("\nIMAGE EXPLANATION:\n", answer)

except Exception as error:
    print("ERROR:", error)



OCR engine: tesseract
Confidence: 27.1%

EXTRACTED TEXT:
 af
— 7 ‘ a, 9 —

IMAGE EXPLANATION:
 The image shows a person standing outdoors, leaning against a low wall or barrier. The person is wearing a blue and red checkered shirt. The background features a large, open area that appears to be a flooded or muddy field, with a mountainous landscape in the distance. The sky is overcast, suggesting a cloudy day. There is a pole or flagpole on the left side of the image. The text "af" and "— 7 ' a, 9 —" is visible in the image, but it is not clear what it represents.


In [15]:
%pip install pytesseract opencv-python ollama


Note: you may need to restart the kernel to use updated packages.


In [18]:
from pathlib import Path
print(Path.cwd())

c:\Users\koda\Downloads


In [20]:
from pathlib import Path
import ollama
import cv2
import pytesseract

# Configure Tesseract OCR on Windows
tesseract = Path(r"C:\Program Files\Tesseract-OCR\tesseract.exe")

if tesseract.exists():
    pytesseract.pytesseract.tesseract_cmd = str(tesseract)


def extract_text(image_path):
    path = Path(image_path.strip('"'))

    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")

    image = cv2.imread(str(path))
    if image is None:
        raise ValueError("Could not open the image.")

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    text = pytesseract.image_to_string(
        gray, config="--oem 3 --psm 6"
    ).strip()

    return text


MODEL = "qwen2.5vl:3b"

def explain_image(image_path, question="Explain this image."):
    path = Path(image_path.strip('"')).resolve()

    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")

    ocr_text = extract_text(str(path))

    response = ollama.chat(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": (
                f"{question}\n\n"
                f"Text detected in the image:\n{ocr_text}\n\n"
                "Explain the image accurately. Do not invent details."
            ),
            "images": [str(path)]
        }]
    )

    return ocr_text, response["message"]["content"]


print("Functions loaded successfully!")

Functions loaded successfully!


In [21]:
image_path = input("Enter the full image path: ").strip().strip('"')
question = input("Ask a question about the image: ").strip()

ocr_text, explanation = explain_image(
    image_path,
    question or "Explain this image."
)

print("\n--- EXTRACTED TEXT ---")
print(ocr_text)

print("\n--- IMAGE EXPLANATION ---")
print(explanation)


--- EXTRACTED TEXT ---
af
— 7 ‘ a, 9 —

--- IMAGE EXPLANATION ---
The image shows a person standing outdoors, leaning against a low wall. The background features a wide, open area with a body of water, possibly a lake or river, and some distant mountains. The person is wearing a blue and red checkered shirt. The sky appears overcast.


In [22]:

from pathlib import Path
import tempfile

import cv2
import ollama
import pytesseract
import streamlit as st
from PIL import Image

MODEL = "qwen2.5vl:3b"

# Change this if Tesseract is installed elsewhere.
TESSERACT_PATH = Path(
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

if TESSERACT_PATH.exists():
    pytesseract.pytesseract.tesseract_cmd = str(TESSERACT_PATH)


def extract_text(image_path):
    image = cv2.imread(str(image_path))

    if image is None:
        return ""

    height, width = image.shape[:2]
    if width < 1400:
        scale = 1400 / width
        image = cv2.resize(
            image,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_CUBIC,
        )

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return pytesseract.image_to_string(
        gray, config="--oem 3 --psm 6"
    ).strip()


st.set_page_config(
    page_title="AI Image Explainer",
    page_icon="🖼️",
    layout="wide",
)

st.title("🖼️ AI Image Explainer")
st.caption("Ask questions about an image using local Ollama AI.")

uploaded = st.file_uploader(
    "Upload an image",
    type=["png", "jpg", "jpeg", "webp"],
)

if uploaded:
    suffix = Path(uploaded.name).suffix or ".png"

    with tempfile.NamedTemporaryFile(
        delete=False, suffix=suffix
    ) as temp:
        temp.write(uploaded.getvalue())
        image_path = Path(temp.name)

    st.image(uploaded, caption=uploaded.name, use_container_width=True)

    if st.button("Extract text"):
        try:
            st.session_state["ocr_text"] = extract_text(image_path)
        except Exception as e:
            st.error(f"OCR error: {e}")

    st.text_area(
        "Extracted text",
        value=st.session_state.get("ocr_text", ""),
        height=150,
        disabled=True,
    )

    if "messages" not in st.session_state:
        st.session_state["messages"] = []

    for message in st.session_state["messages"]:
        with st.chat_message(message["role"]):
            st.write(message["content"])

    question = st.chat_input("Ask a question about this image...")

    if question:
        st.session_state["messages"].append(
            {"role": "user", "content": question}
        )

        with st.chat_message("user"):
            st.write(question)

        try:
            ocr_text = st.session_state.get("ocr_text", "")

            response = ollama.chat(
                model=MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": (
                            f"Question: {question}\n\n"
                            f"Extracted text (may contain errors):\n"
                            f"{ocr_text}\n\n"
                            "Answer based on the image. Be accurate "
                            "and mention uncertainty when appropriate."
                        ),
                        "images": [str(image_path.resolve())],
                    }
                ],
            )

            answer = response["message"]["content"]

            st.session_state["messages"].append(
                {"role": "assistant", "content": answer}
            )

            with st.chat_message("assistant"):
                st.write(answer)

        except Exception as e:
            st.error(
                f"AI error: {e}. Check that Ollama is running "
                f"and the model '{MODEL}' is installed."
            )
else:
    st.info("Upload an image to begin.")


2026-10-21 21:39:05.400 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-21 21:39:05.404 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-21 21:39:05.546 
  command:

    streamlit run c:\Users\koda\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-10-21 21:39:05.547 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-21 21:39:05.547 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-21 21:39:05.548 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-21 21:39:05.550 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runni